# KyTea で単語分割と読みの推定をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/kytea.ipynb)

NLP クックブック 03-21 ／ 日本語の形態素解析と文分割

辞書の照合ではなく、文字の並びから「ここで単語が切れるか」を判定する解析器で、単語分割と品詞・読みを得たい。辞書にない語の読みも推定できる KyTea（京都テキスト解析ツールキット）を使う。

In [1]:
!pip install -q kytea fugashi unidic-lite
# 学習済みの日本語モデル (gzip で約 38MB、展開後 約 120MB) はソースのリポジトリに同梱されている。pip のパッケージには含まれない
!wget -q https://github.com/neubig/kytea/raw/master/data/model.bin.gz && gunzip -f model.bin.gz && ls -lh model.bin

In [2]:
import Mykytea

kytea = Mykytea.Mykytea("-model model.bin")   # オプションはコマンド版の kytea と同じ書き方で渡す
s = "太郎は図書館で借りた本を花子に渡した。"
print(" ".join(kytea.getWS(s)))      # 単語分割だけ
print(kytea.getTagsToString(s))      # 単語/品詞/読み

# getTags は単語ごとに [品詞の候補, 読みの候補] を返す (各候補は (タグ, スコア))
for w in kytea.getTags(s):
    (pos, pos_score), = w.tag[0]
    (yomi, yomi_score), = w.tag[1]
    print(w.surface, pos, yomi, round(pos_score, 2), round(yomi_score, 2), sep="\t")

太郎 は 図書 館 で 借り た 本 を 花子 に 渡 し た 。
太郎/名詞/たろう は/助詞/は 図書/名詞/としょ 館/接尾辞/かん で/助詞/で 借り/動詞/かり た/助動詞/た 本/名詞/ほん を/助詞/を 花子/名詞/はなこ に/助詞/に 渡/動詞/わた し/語尾/し た/助動詞/た 。/補助記号/。 
太郎	名詞	たろう	2.93	100.0
は	助詞	は	3.28	100.0
図書	名詞	としょ	2.9	100.0
館	接尾辞	かん	2.67	1.0
で	助詞	で	2.29	1.0
借り	動詞	かり	2.08	100.0
た	助動詞	た	2.11	100.0
本	名詞	ほん	2.14	1.99
を	助詞	を	2.57	2.0
花子	名詞	はなこ	2.78	100.0
に	助詞	に	2.16	2.0
渡	動詞	わた	2.55	100.0
し	語尾	し	2.72	1.0
た	助動詞	た	2.61	100.0
。	補助記号	。	2.63	100.0


In [3]:
# getAllTags は 2 位以下の候補とスコアも返す (-tagmax で候補数を変えられる。既定は 3)。読みは文脈で変わる
for s in ["日本語の本を今日読んだ。", "今日は日本語の本を読んだ。"]:
    for w in kytea.getAllTags(s):
        if w.surface == "日本":
            print(s, "日本 の読みの候補:", [(t, round(sc, 2)) for t, sc in w.tag[1]])

# -notags は分割だけ、-notag 1 は 1 番目のタグ (品詞) を推定せず読みだけを付ける
print(Mykytea.Mykytea("-model model.bin -notags").getTagsToString("日本語の本を今日読んだ。"))
print(Mykytea.Mykytea("-model model.bin -notag 1").getTagsToString("日本語の本を今日読んだ。"))

日本語の本を今日読んだ。 日本 の読みの候補: [('にほん', 0.08), ('にっぽん', 0.0)]
今日は日本語の本を読んだ。 日本 の読みの候補: [('にっぽん', 0.54), ('にほん', 0.0)]
日本 語 の 本 を 今日 読 ん だ 。 
日本/にほん 語/ご の/の 本/ほん を/を 今日/きょう 読/よ ん/ん だ/だ 。/。 


In [4]:
# MeCab + UniDic (fugashi) と分割の単位を比べる
from fugashi import Tagger

tagger = Tagger()
for s in ["食べさせられたくなかった。", "大規模言語モデルを用いた検索拡張生成の論文を読んだ。", "ググってもわからんかった。"]:
    print("KyTea  :", " | ".join(kytea.getWS(s)))
    print("UniDic :", " | ".join(w.surface for w in tagger(s)))

KyTea  : 食べ | させ | られ | た | く | な | かっ | た | 。
UniDic : 食べ | させ | られ | たく | なかっ | た | 。
KyTea  : 大 | 規模 | 言語 | モデル | を | 用い | た | 検索 | 拡張 | 生成 | の | 論文 | を | 読 | ん | だ | 。
UniDic : 大 | 規模 | 言語 | モデル | を | 用い | た | 検索 | 拡張 | 生成 | の | 論文 | を | 読ん | だ | 。
KyTea  : ググ | って | も | わか | ら | ん | かっ | た | 。
UniDic : ググっ | て | も | わから | んかっ | た | 。


In [5]:
# UniDic (unidic-lite) で未知語になったり細かく分かれたりする語を含む文 (最後の文は「辞書にない未知語を見つける」と同じ)。
# UniDic は未知語に読みを付けられないが、KyTea は自分の辞書にない語 (「鬼滅」「ジーピーティー」) も文字の並びから区切りと読みを推定する
for s in ["鬼滅の刃の竈門炭治郎が好きだ。", "チャットジーピーティーに聞いた。", "彼はスマホをポチポチいじっていた。", "推し活でエモい写真をインスタにあげた。"]:
    print("KyTea  :", kytea.getTagsToString(s))
    print("UniDic :", " ".join(f"{w.surface}/{'(未知語)' if w.is_unk else w.feature.kana or w.surface}" for w in tagger(s)))

KyTea  : 鬼滅/名詞/きめつ の/助詞/の 刃/名詞/は の/助詞/の 竈門/名詞/かまど 炭/名詞/たん 治郎/名詞/じろう が/助詞/が 好き/形状詞/すき だ/助動詞/だ 。/補助記号/。 
UniDic : 鬼/オニ 滅/メツ の/ノ 刃/ハ の/ノ 竈門/カマド 炭/タン 治郎/ジロウ が/ガ 好き/スキ だ/ダ 。/。
KyTea  : チャット/名詞/ちゃっと ジーピーティー/名詞/じーぴーてぃー に/助詞/に 聞/動詞/き い/語尾/い た/助動詞/た 。/補助記号/。 
UniDic : チャットジーピーティー/(未知語) に/ニ 聞い/キイ た/タ 。/。
KyTea  : 彼/代名詞/かれ は/助詞/は スマホ/名詞/すまほ を/助詞/を ポチポチ/名詞/ぽちぽち いじ/動詞/いじ っ/語尾/っ て/助詞/て い/動詞/い た/助動詞/た 。/補助記号/。 
UniDic : 彼/カレ は/ハ スマホ/(未知語) を/ヲ ポチポチ/ポチポチ いじっ/イジッ て/テ い/イ た/タ 。/。
KyTea  : 推/動詞/お し活/名詞/しかつ で/助詞/で エモ/形容詞/えも い/語尾/い 写真/名詞/しゃしん を/助詞/を インスタ/名詞/いんすた に/助詞/に あげ/動詞/あげ た/助動詞/た 。/補助記号/。 
UniDic : 推し/オシ 活/カツ で/デ エ/エ モ/モ い/イ 写真/シャシン を/ヲ イン/イン スタ/スタ に/ニ あげ/アゲ た/タ 。/。


In [6]:
# 速度の比較 (10,000 文)
import time

texts = ["国境の長いトンネルを抜けると雪国であった。", "自然言語処理の小ネタを集めたクックブックを作りたい。"] * 5000
for name, f in [("KyTea (分割のみ)", Mykytea.Mykytea("-model model.bin -notags").getWS),
                ("KyTea (品詞・読み)", kytea.getTags),
                ("MeCab + UniDic", tagger)]:
    t = time.perf_counter()
    for x in texts:
        f(x)
    print(f"{name:16} {time.perf_counter() - t:5.2f} 秒")

KyTea (分割のみ)      0.15 秒
KyTea (品詞・読み)     0.82 秒
MeCab + UniDic    0.11 秒


- KyTea は Graham Neubig らが開発した解析器。文字と文字の間ごとに単語境界かどうかを、周りの文字 n-gram・文字種・辞書との一致を素性とする線形分類器（線形 SVM やロジスティック回帰。学習は LIBLINEAR）で 1 か所ずつ独立に判定し、単語の品詞と読みも個別に推定する（点予測）。付録「仕組みを理解するための実装」の「単語分割を文字単位の系列ラベリングとして解く」の CRF はラベル列全体をまとめて決めるが、KyTea は隣の判定結果を使わないので、一部の境界や読みだけを付けた「部分的アノテーション」のデータでも学習できる。
- 同梱のモデルは現代日本語書き言葉均衡コーパス (BCCWJ) や UniDic などで学習されたもので、活用語尾を語幹と別の単語にする（「渡 し た」「読 ん だ」「た く な かっ た」）。MeCab + UniDic の短単位（「読ん」「たく」「なかっ」）より細かい。構文解析の章の「Ckylark で日本語の句構造解析をする」の日本語モデルは、この分割を前提にしている。
- 読みはひらがなで、単語の表記に対応する部分だけが付く（「渡」→「わた」）。送り仮名が別の単語になるので、この章の「漢字の読みを取得してふりがな（ルビ）を付ける」（送り仮名ごとルビが付く）と違い、漢字の部分だけにルビを付けやすい。辞書で読みの候補が 1 つしかない語はスコアが 100 になり、「日本」のように候補が複数ある語は文脈で結果が変わる（「にほん」と「にっぽん」）。
- 辞書にない語の推定は誤ることもある（「推し活」は「推（お）/ し活」に分割された）。Python のバインディングからは未知語かどうかは分からない。コマンド版の `kytea -unktag '[UNK]'` で同じ文を試すと、「鬼滅」「ジーピーティー」「し活」「エモ」「ポチポチ」に印が付き、「スマホ」「インスタ」は KyTea の辞書にある語だった。
- 検証環境で 10,000 文にかかった時間は、品詞・読みありが 0.8〜0.9 秒、分割だけ (`-notags`) が約 0.15 秒、MeCab + UniDic が 0.1 秒ほど。モデルの読み込みに約 2 秒かかるので、`Mykytea` は 1 回だけ作って使い回す。
- pip の `kytea` (Mykytea-python) は KyTea のライブラリを含むのでビルドは不要だが、モデルは含まない。モデルは研究・商用での利用は許可されているが無断の再配布は禁じられているので、各自が作者のリポジトリから取得する。コマンド版や学習用の `train-kytea`（`-part` で部分的アノテーション、`-dict` で辞書を与える）が要るなら、構文解析の章の「Ckylark で日本語の句構造解析をする」のようにソースからビルドする。ソースは Apache License 2.0 で、最後のリリースは 0.4.7（2014 年）、リポジトリの最後の更新は 2020 年。